<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/colab20260606.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1: 基础环境与插件管理器 (内置通用安装函数)
# ==========================================
import os, subprocess
print("=== 🚀 开始安装 ComfyUI 基础与图像模块 ===")
%cd /content

if not os.path.exists("ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    !cd ComfyUI && git pull

%cd /content/ComfyUI
!pip install -q -r requirements.txt huggingface_hub hf_transfer

# 定义通用节点安装函数以复用代码
def install_node(repo_url):
    folder_name = repo_url.split('/')[-1].replace('.git', '')
    target_path = f"/content/ComfyUI/custom_nodes/{folder_name}"
    if not os.path.exists(target_path):
        !git clone {repo_url} {target_path}
        if os.path.exists(f"{target_path}/requirements.txt"):
            !pip install -q -r {target_path}/requirements.txt
    else:
        !cd {target_path} && git pull

# 安装基础核心插件
base_nodes = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/rgthree/rgthree-comfy.git"
]
for node in base_nodes:
    install_node(node)
print("\n✅ Cell 1 安装完成！")

ALL-ONE-IN最全脚本

In [ ]:
# ==========================================
# 终极节点自动化安装脚本（修复 ComfyMath + LTXVideo依赖 + LayerStyle + EasyUse + Masquerade + RES4LYF）
# ==========================================
import os
import subprocess

# 请确认这是你的 ComfyUI 根目录路径（Colab / 云实例通常是这个）
comfyui_dir = "/content/ComfyUI"
custom_nodes_dir = os.path.join(comfyui_dir, "custom_nodes")

# 修正后的节点清单
node_repos = [
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Lightricks/ComfyUI-LTXVideo.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/cubiq/ComfyUI_essentials.git",
    "https://github.com/yuvraj108c/ComfyUI-Video-Depth-Anything.git",
    #"https://github.com/ai-shizuka/comfyui-tbox.git",
    "https://github.com/evanspearman/ComfyMath.git",
    #"https://github.com/Smirnov75/ComfyUI-mxToolkit.git",
    "https://github.com/chrisgoringe/cg-use-everywhere.git",

    "https://github.com/rgthree/rgthree-comfy.git",

    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    #"https://github.com/city96/ComfyUI-GGUF.git",
    "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git",
    "https://github.com/WASasquatch/was-node-suite-comfyui.git",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "https://github.com/DoctorDiffusion/ComfyUI-MediaMixer.git",
    "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",
    "https://github.com/11cafe/comfyui-workspace-manager.git",
    "https://github.com/kijai/ComfyUI-PromptRelay.git",

    "https://github.com/Saganaki22/ComfyUI-FishAudioS2.git",
    "https://github.com/WhatDreamsCost/WhatDreamsCost-ComfyUI.git",

    # 修复上一轮缺失的节点
    "https://github.com/chflame163/ComfyUI_LayerStyle.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",

    # ==========================================
    # ▼ 本次新增：修复 Image To Mask 节点 ▼
    # ==========================================
    "https://github.com/BadCafeCode/masquerade-nodes-comfyui.git",

    # ==========================================
    # ▼ 本次新增：ClownSampler 节点 (位于 RES4LYF 包中) ▼
    # ==========================================
    "https://github.com/ClownsharkBatwing/RES4LYF.git"
]

print("=== 🚀 开始自动化下载并安装节点及其依赖 ===")

os.makedirs(custom_nodes_dir, exist_ok=True)

for repo in node_repos:
    repo_name = repo.split('/')[-1].replace('.git', '')
    repo_path = os.path.join(custom_nodes_dir, repo_name)

    if not os.path.exists(repo_path):
        print(f"\n📥 正在克隆: {repo_name}...")
        result = subprocess.run(["git", "clone", repo, repo_path], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"   ❌ 克隆失败: {result.stderr}")
        else:
            print(f"   ✅ 克隆成功")
    else:
        print(f"\n✅ 已存在: {repo_name}，正在尝试拉取更新...")
        pull_result = subprocess.run(["git", "pull"], cwd=repo_path, capture_output=True, text=True)
        if pull_result.returncode != 0:
            print(f"   ⚠️ 更新失败 (可能是本地有冲突，请手动检查): {pull_result.stderr.strip()}")
        else:
            print(f"   🔄 代码已是最新或更新成功")

    # 安装依赖
    req_file = os.path.join(repo_path, "requirements.txt")
    if os.path.exists(req_file):
        print(f"   ⚙️ 安装 {repo_name} 的依赖...")
        subprocess.run(["pip", "install", "-r", req_file, "--quiet"])
    else:
        print(f"   ⏭️ {repo_name} 无 requirements.txt")

# ==========================================
# 🔧 针对 LTXVideo 的 Kornia 冲突修复补丁
# ==========================================
print("\n=== 🔧 执行特定节点依赖修复 (ComfyUI-LTXVideo) ===")
print("   ⬇️ 强制降级 kornia==0.7.2...")
subprocess.run(["pip", "install", "kornia==0.7.2", "--quiet"])

ltx_req = os.path.join(custom_nodes_dir, "ComfyUI-LTXVideo", "requirements.txt")
if os.path.exists(ltx_req):
    print("   ⚙️ 重新校验 LTXVideo 依赖树...")
    subprocess.run(["pip", "install", "-r", ltx_req, "--quiet"])
else:
    print("   ⏭️ 未找到 LTXVideo requirements.txt，跳过校验")

# ==========================================
# 全局常用依赖
# ==========================================
print("\n=== 📦 安装全局常用库 ===")
subprocess.run(["pip", "install", "--quiet", "opencv-python-headless", "scikit-image", "onnxruntime-gpu", "sageattention", "gguf"])

print("\n🎉 所有节点安装及更新完成！请重启 ComfyUI（Restart）")

In [ ]:
# ==========================================
# 单独下载 Fish Audio S2 (s2-pro-fp8) 模型
# ==========================================
import os
from huggingface_hub import snapshot_download

# 设定 ComfyUI 的根目录（如果是本地环境，请修改为你的实际路径）
comfyui_dir = "/content/ComfyUI"

# 根据 Fish Audio S2 节点的要求，设定标准存放路径
target_dir = os.path.join(comfyui_dir, "models", "FishAudioS2", "s2-pro-fp8")

print(f"🚀 开始下载 s2-pro-fp8 模型...")
print(f"📂 目标安装路径: {target_dir}")

try:
    # 开启 HF 传输加速（可选）
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    # 下载整个模型仓库到指定文件夹
    snapshot_download(
        repo_id="drbaph/s2-pro-fp8",
        local_dir=target_dir,
        local_dir_use_symlinks=False  # 禁用软链接，确保文件实实在在地存在于该目录
    )
    print("\n🎉 s2-pro-fp8 模型下载并安装成功！")
    print("👉 请重启 ComfyUI，并在 FishS2MultiSpeakerTTS 节点的 model_path 菜单中选择 's2-pro-fp8'。")

except Exception as e:
    print(f"\n❌ 下载失败，错误信息: {e}")

In [ ]:
# ==========================================
# Cell 3: 终极模型多线程极速同步 (25合1 防冲突&防重下 完美版)
# ==========================================
import os
import shutil
from google.colab import userdata
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except:
    raise Exception("❌ 请在左侧 '🔑 Secrets' 中添加 'HF_TOKEN'")

downloads = [
    # --- 1. 基础核心模型 ---
   # {"repo_id": "Lightricks/LTX-2.3-fp8", "filename": "ltx-2.3-22b-dev-fp8.safetensors", "final_dir": "/content/ComfyUI/models/checkpoints"},
    {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-22b-distilled-lora-384-1.1.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "final_dir": "/content/ComfyUI/models/latent_upscale_models"},
    {"repo_id": "Comfy-Org/ltx-2", "filename": "split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors", "final_dir": "/content/ComfyUI/models/text_encoders"},
    {"repo_id": "Comfy-Org/ltx-2.3", "filename": "split_files/loras/ltx-2.3-id-lora-talkvid-3k.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-Union-Control", "filename": "ltx-2.3-22b-ic-lora-union-control-ref0.5.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3", "filename": "ltx-2.3-22b-distilled-1.1.safetensors", "final_dir": "/content/ComfyUI/models/checkpoints"},

    # --- 2. 社区增强与特征映射模型 ---
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "text_encoders/ltx-2.3_text_projection_bf16.safetensors", "final_dir": "/content/ComfyUI/models/text_encoders"},
    {"repo_id": "stronman/LTX-2.3-Transition-LORA", "filename": "ltx2.3-transition.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "stronman/Ltx2.3-VBVR-lora-I2V", "filename": "Ltx2.3-Licon-VBVR-I2V-390K-R32.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "loras/ltx-2.3-22b-distilled-lora-dynamic_fro09_avg_rank_105_bf16.safetensors", "final_dir": "/content/ComfyUI/models/loras", "flatten": True},

    # --- 3. LTX 官方功能性特效 LoRA ---
    {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-HDR", "filename": "ltx-2.3-22b-ic-lora-hdr-0.9.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-Motion-Track-Control", "filename": "ltx-2.3-22b-ic-lora-motion-track-control-ref0.5.safetensors", "final_dir": "/content/ComfyUI/models/loras"},
    {"repo_id": "Lightricks/LTX-2.3-22b-IC-LoRA-LipDub", "filename": "ltx-2.3-22b-ic-lora-lipdub-0.9.safetensors", "final_dir": "/content/ComfyUI/models/loras"},

    # --- 4. IP-Adapter 与 图像特征模型 ---
    {"repo_id": "h94/IP-Adapter-FaceID", "filename": "ip-adapter-faceid-plusv2_sdxl.bin", "final_dir": "/content/ComfyUI/models/ipadapter"},
    {"repo_id": "h94/IP-Adapter", "filename": "models/image_encoder/model.safetensors", "final_dir": "/content/ComfyUI/models/clip_vision", "flatten": True, "rename_to": "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"},

    # --- 5. 路径纠偏与重命名修复模型 ---
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "diffusion_models/ltx-2.3-22b-distilled-1.1_transformer_only_fp8_scaled.safetensors", "final_dir": "/content/ComfyUI/models/unet", "flatten": True},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/LTX23_video_vae_bf16.safetensors", "final_dir": "/content/ComfyUI/models/vae", "flatten": True},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/LTX23_audio_vae_bf16.safetensors", "final_dir": "/content/ComfyUI/models/vae", "flatten": True},

    # ⬇️ ICEdit-Insight 视频特效全家桶 (自动追加 -lora 后缀) ⬇️
    {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-watermark-remove-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-watermark-remove-general-lora.safetensors"},
    {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-subtitles-remove-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-subtitles-remove-general-lora.safetensors"},
    # [新增] 视频修复模型
    {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-video-restoration-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-video-restoration-general-lora.safetensors"},
    # [新增] 视频高清放大增强模型
    {"repo_id": "joyfox/LTX2.3-ICEdit-Insight", "filename": "ltx2.3-ic-video-upscale-general.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "ltx2.3-ic-video-upscale-general-lora.safetensors"},

    # --- 6. 额外补充模型 ---
    {"repo_id": "Muapi/ltx2.3-resolution-enhancement-guofeng", "filename": "ltx2.3-resolution-enhancement-guofeng.safetensors", "final_dir": "/content/ComfyUI/models/loras", "rename_to": "LTX-2.3-Guofeng-V0.1.safetensors"},
    {"repo_id": "Kijai/LTX2.3_comfy", "filename": "vae/taeltx2_3.safetensors", "final_dir": "/content/ComfyUI/models/vae", "flatten": True}
]

def download_model(task):
    try:
        # 建立最终目标文件夹
        os.makedirs(task["final_dir"], exist_ok=True)

        # 1. 提前计算该模型最终期望的绝对路径位置
        base_name = task.get("rename_to", os.path.basename(task["filename"]))
        final_path = os.path.join(task["final_dir"], base_name)

        # 💡 【核心防重下逻辑】：如果最终位置已经有这个文件了，直接跳过下载！
        if os.path.exists(final_path):
            print(f"⏩ 已存在，跳过下载: {base_name}")
            return

        # 2. 如果不存在，再向 HuggingFace 请求下载
        downloaded_path = hf_hub_download(
            repo_id=task["repo_id"],
            filename=task["filename"],
            local_dir=task["final_dir"],
            repo_type="model"
        )

        # 3. 纠偏剪切：处理多层级空文件夹和重命名
        if ("flatten" in task or "rename_to" in task) and downloaded_path != final_path:
            os.makedirs(os.path.dirname(final_path), exist_ok=True)
            if os.path.exists(final_path):
                os.remove(final_path)
            shutil.move(downloaded_path, final_path)

        print(f"✅ 成功就绪: {base_name}")
    except Exception as e:
        print(f"❌ 失败 {task['filename'].split('/')[-1]}: {e}")

print(f"🚀 开始启动多线程并发下载与智能分类，共 {len(downloads)} 个模型...")

# 使用 8 线程并发下载
with ThreadPoolExecutor(max_workers=8) as executor:
    executor.map(download_model, downloads)

print("\n🎉 全量模型并发同步 & 路径纠偏全部完成！")

In [ ]:
# ==========================================
# Cell 4: FRP 内网穿透配置 (流式解压极速部署)
# ==========================================
import os
from google.colab import userdata

try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except:
    raise Exception("❌ 请检查 'VPS_IP' 和 'FRP_TOKEN' 密钥是否配置！")

# 极速流式下载解压
if not os.path.exists("/content/frp_0.56.0_linux_amd64"):
    !wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content

frpc_conf = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
"""
with open("/content/frp_0.56.0_linux_amd64/frpc.toml", "w") as f:
    f.write(frpc_conf.strip())

print("✅ FRP 极速部署并配置完毕！")

In [8]:
# ==========================================
# Cell 4: 启动 FRP 和 ComfyUI (重启界面时运行),不注册，启动快！访问域名
# ==========================================
import subprocess
import threading
import os
import configparser

import time

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
# --- 新增：防断开保活机制 ---
def keep_alive():
    while True:
        time.sleep(300)  # 每 5 分钟
        print("\n[Keep-Alive] 保持 Colab 连接活跃中...")

print("⏳ 正在启动防断开后台保活线程...")
threading.Thread(target=keep_alive, daemon=True).start()
# --------------------------

# 1. 在后台新线程启动 FRP
def start_frpc():
    subprocess.run(["/content/frp_0.56.0_linux_amd64/frpc", "-c", "/content/frp_0.56.0_linux_amd64/frpc.toml"])

print("⏳ 正在后台唤起 FRP 穿透服务...")
threading.Thread(target=start_frpc, daemon=True).start()

print("============================================================")
print("✅ FRP 穿透已就绪！")
# 这里已修改为你的指定域名（保留了 8080 端口，如果你的服务端用 Nginx 做了 80 端口反代，可以把 :8080 删掉）
print("👉 启动完成后，请通过浏览器访问: http://cjp.usdream.dpdns.org:8090")
print("============================================================\n")

# ---------------------------------------------------------
# 新增：彻底拦截 ComfyUI-Manager 启动时的联网 Fetch 行为
# ---------------------------------------------------------
print("⏳ 正在配置 Manager 网络模式以跳过 Fetch...")
# 兼容所有版本的配置文件路径 (特别是最新的 __manager 路径)
manager_config_paths = [
    "/content/ComfyUI/user/__manager/config.ini",               # 最新版 V3.39+ 配置路径
    "/content/ComfyUI/user/default/ComfyUI-Manager/config.ini", # 较新版配置路径
    "/content/ComfyUI/custom_nodes/ComfyUI-Manager/config.ini"  # 老旧版配置路径
]

for config_path in manager_config_paths:
    os.makedirs(os.path.dirname(config_path), exist_ok=True)
    config = configparser.ConfigParser()

    if os.path.exists(config_path):
        config.read(config_path)

    if 'default' not in config:
        config['default'] = {}

    # 将网络模式改为 private
    config['default']['network_mode'] = 'private'

    with open(config_path, 'w') as f:
        config.write(f)

print("✅ 已成功切断 Fetch ComfyRegistry Data 流程！\n")
# ---------------------------------------------------------

# 2. 启动 ComfyUI 主程序
print("⏳ 正在启动 ComfyUI 主进程...\n")
%cd /content/ComfyUI
!python main.py --dont-print-server


[Keep-Alive] 保持 Colab 连接活跃中...
[INFO] 
Stopped server
^C
